# AI Workforce Statements: Historical Firm Crosswalk

## Part 0 Notebook Setup and Research Scope

**Purpose.** Prepare a transparent historical identifier crosswalk for firms represented
in the S&P 500 from **January 1, 2018 through December 31, 2025**.
The intended identifier relationships are:

**CRSP PERMNO / PERMCO / CCM / Compustat GVKEY / CIK / Revelio RCID**

This notebook currently contains only setup and initial input diagnostics. It does not
construct matches, select preferred records, change membership dates, or export data.
Each subsequent Part requires researcher review and approval.

### Sources and units of observation

| Source | Current unit of observation | Relevant identifiers |
|---|---|---|
| CRSP historical S&P 500 extract | Daily security observations with membership dates | PERMNO, PERMCO, CUSIP, ticker |
| CCM link history (`ccmxpf_lnkhist`, as documented in the handoff) | Dated security-to-company link records | LPERMNO, LPERMCO, GVKEY |
| Compustat Fundamentals Annual | Annual company observations, potentially with multiple formats | GVKEY, CIK, CUSIP |
| Revelio company mapping | Candidate company mappings retrieved using historical tickers | RCID, GVKEY, CIK, CUSIP |

**PERMNO** identifies a CRSP security. 

**PERMCO** identifies a CRSP company.

Multiple securities can belong to one company. 

**GVKEY** identifies a Compustat company.

**CIK** identifies an SEC filer. 

**RCID** identifies a Revelio entity. 

**CUSIP** can refer to a security issue.

**ticker** is not a reliable company identifier.


**Historical validity matters.** A candidate identifier match alone does not establish
that two records refer to the same entity during the relevant membership period.
CCM link dates, corporate changes, and parent/subsidiary relationships require review.
The Revelio extract is a candidate mapping; its parent hierarchy is not assumed to be
historically valid throughout the study period.

**Data handling.** Keep licensed records local. Display aggregate diagnostics in this
Part, preserve source files, and do not export results. Before any future GitHub upload,
the researcher must inspect and clear proprietary record-level notebook outputs.

In [2]:
from pathlib import Path

import pandas as pd
from IPython.display import display

study_start = pd.Timestamp("2018-01-01")
study_end = pd.Timestamp("2025-12-31")

# Support execution from the project root, code/, or code/notebooks/.
for project_root in [Path("."), Path(".."), Path("../..")]:
    if (project_root / "data" / "raw").is_dir():
        break
else:
    raise FileNotFoundError("Run from the project root, code/, or code/notebooks/.")

print(f"Study period: {study_start.date()} to {study_end.date()}")
print(f"pandas version: {pd.__version__}")

Study period: 2018-01-01 to 2025-12-31
pandas version: 3.0.1


## Part 1 Data Inventory and Initial Quality Checks

### 1.1 Input inventory




In [3]:
input_paths = {
    "crsp": project_root / "data/raw/crsp/crsp_sp500_20261022.csv",
    "ccm": project_root / "data/raw/ccm/ccm_link_history_20261023.csv",
    "compustat": project_root / "data/raw/compustat/Compustat Fundamentals Annual.csv",
    "revelio": project_root / "data/raw/revelio/revelio_company_mapping_20261023.csv",
}

intermediate_paths = {
    "membership_intervals": project_root / "data/clean/sp500_membership_intervals.csv",
    "permno_query_aid": project_root / "data/clean/sp500_permno_for_ccm.txt",
    "ticker_query_aid": project_root / "data/clean/sp500_tickers_for_revelio.txt",
}

inventory_rows = []
for name, path in {**input_paths, **intermediate_paths}.items():
    inventory_rows.append({
        "input": name,
        "relative_path": path.relative_to(project_root).as_posix(),
        "exists": path.is_file(),
        "size_mb": path.stat().st_size / 1_000_000 if path.is_file() else pd.NA,
    })

inventory = pd.DataFrame(inventory_rows)
display(inventory.round({"size_mb": 2}))

if not inventory["exists"].all():
    raise FileNotFoundError("An expected input is missing. Review the inventory above.")

,input,relative_path,exists,size_mb
0,crsp,data/raw/crsp/crsp_sp500_20261022.csv,True,305.54
1,ccm,data/raw/ccm/ccm_link_history_20261023.csv,True,0.03
2,compustat,data/raw/compustat/Compustat Fundamentals Annu...,True,10.85
3,revelio,data/raw/revelio/revelio_company_mapping_20261...,True,0.25
4,membership_intervals,data/clean/sp500_membership_intervals.csv,True,0.03
5,permno_query_aid,data/clean/sp500_permno_for_ccm.txt,True,0.00
6,ticker_query_aid,data/clean/sp500_tickers_for_revelio.txt,True,0.00


### 1.2 Source schemas and reading choices

Read each CSV header before loading data. The CRSP file contains many trading variables;
only nine membership and identifier fields are needed for this initial inspection.
All columns in the other three raw files are read.

Identifiers such as **GVKEY, CIK, and CUSIP are strings**, not quantities. String storage
preserves leading zeros and alphanumeric characters. Dates and numeric variables are
also initially retained as strings so that source encodings remain visible. Temporary
date conversions below are used only for diagnostics.

The reading rule treats an empty CSV field as missing, but disables pandas' broader
default missing-token list. A literal `NA` may be a ticker, so it must not be silently
converted to a missing identifier. Other possible missing tokens are counted separately.
Column names are lowercased in memory; source files are unchanged.

In [4]:
source_columns = {}
for name, path in input_paths.items():
    source_columns[name] = pd.read_csv(path, nrows=0).columns.tolist()
    print(f"{name}: {len(source_columns[name])} source columns")
    print(", ".join(source_columns[name]))
    print()

crsp: 42 source columns
INDNO, MbrStartDt, MbrEndDt, PERMNO, HdrCUSIP, PERMCO, SICCD, CUSIP, Ticker, ShareType, SecurityType, SecuritySubType, USIncFlg, IssuerType, PrimaryExch, ConditionalType, TradingStatusFlg, DlyCalDt, DlyPrc, DlyPrcFlg, DlyCap, DlyCapFlg, DlyRet, DlyRetx, DlyRetMissFlg, DlyRetDurFlg, DlyOrdDivAmt, DlyNonOrdDivAmt, DlyFacPrc, DlyDistRetFlg, DlyVol, DlyClose, DlyLow, DlyHigh, DlyBid, DlyAsk, DlyOpen, DlyNumTrd, DlyMMCnt, DlyCumFacPr, DlyCumFacShr, ShrOut

ccm: 8 source columns
GVKEY, LINKPRIM, LIID, LINKTYPE, LPERMNO, LPERMCO, LINKDT, LINKENDDT

compustat: 15 source columns
costat, curcd, datafmt, indfmt, consol, tic, datadate, gvkey, conm, cusip, cik, naics, sic, fyear, emp

revelio: 16 source columns
ticker, rcid, company, factset_entity_id, isin, cusip, url, naics_code, cik, lei, linkedin_url, child_rcid, child_company, ultimate_parent_rcid, ultimate_parent_company_name, gvkey



In [5]:
crsp_columns = [
    "INDNO", "MbrStartDt", "MbrEndDt", "PERMNO", "PERMCO",
    "CUSIP", "HdrCUSIP", "Ticker", "DlyCalDt",
]

crsp = pd.read_csv(
    input_paths["crsp"], usecols=crsp_columns,
    dtype="string", keep_default_na=False, na_values=[""],
)
ccm = pd.read_csv(
    input_paths["ccm"], dtype="string", keep_default_na=False, na_values=[""],
)
compustat = pd.read_csv(
    input_paths["compustat"], dtype="string", keep_default_na=False, na_values=[""],
)
revelio = pd.read_csv(
    input_paths["revelio"], dtype="string", keep_default_na=False, na_values=[""],
)

datasets = {
    "crsp": crsp,
    "ccm": ccm,
    "compustat": compustat,
    "revelio": revelio,
}
for frame in datasets.values():
    frame.columns = frame.columns.str.lower()

shape_rows = []
for name, frame in datasets.items():
    shape_rows.append({
        "dataset": name,
        "rows": len(frame),
        "source_columns": len(source_columns[name]),
        "loaded_columns": len(frame.columns),
    })

display(pd.DataFrame(shape_rows))

,dataset,rows,source_columns,loaded_columns
0,crsp,1013723,42,9
1,ccm,788,8,8
2,compustat,107179,15,15
3,revelio,1124,16,16


### 1.3 Identifier counts

Counts below use nonmissing, nonblank values after trimming surrounding whitespace
**for diagnostics only**. Source values in the loaded tables are not replaced. Leading
zeros are preserved, and no cross-source identifier normalization is performed.
Multiple rows per identifier may be expected given each source's unit of observation.

In [6]:
identifier_columns = {
    "crsp": ["indno", "permno", "permco", "cusip", "hdrcusip", "ticker"],
    "ccm": ["lpermno", "lpermco", "gvkey"],
    "compustat": ["gvkey", "cik", "cusip", "tic"],
    "revelio": ["rcid", "gvkey", "cik", "cusip", "ticker"],
}

identifier_rows = []
for name, columns in identifier_columns.items():
    for column in columns:
        values = datasets[name][column].str.strip()
        present = values.notna() & values.ne("")
        identifier_rows.append({
            "dataset": name,
            "identifier": column,
            "rows_with_value": int(present.sum()),
            "rows_missing_or_blank": int((~present).sum()),
            "unique_values": values.loc[present].nunique(),
        })

display(pd.DataFrame(identifier_rows))

,dataset,identifier,rows_with_value,rows_missing_or_blank,unique_values
0,crsp,indno,1013723,0,1
1,crsp,permno,1013723,0,664
2,crsp,permco,1013723,0,657
3,crsp,cusip,1013692,31,705
4,crsp,hdrcusip,1013723,0,664
5,crsp,ticker,1013692,31,697
6,ccm,lpermno,788,0,664
7,ccm,lpermco,788,0,657
8,ccm,gvkey,788,0,674
9,compustat,gvkey,107179,0,18502


### 1.4 Missing values and possible missing encodings

For each loaded column, report empty-field missing values separately from whitespace-only
strings. The percentage combines those two categories and uses **all rows of that
source** as its denominator. CRSP diagnostics cover only the nine loaded fields.

The final column flags strings that can sometimes encode missing data. These are
**candidates for review, not confirmed missing values**, and are not recoded. This list
is not exhaustive. `E` is deliberately excluded: it represents an open-ended CCM link.

In [7]:
possible_missing_tokens = [".", "NA", "N/A", "NULL", "null", "NaN", "nan", "None"]

for name, frame in datasets.items():
    missing_rows = []
    for column in frame.columns:
        values = frame[column]
        stripped = values.str.strip()
        empty_fields = int(values.isna().sum())
        whitespace_only = int(stripped.eq("").sum())
        missing_rows.append({
            "column": column,
            "empty_fields": empty_fields,
            "whitespace_only": whitespace_only,
            "missing_or_blank_pct": (
                100 * (empty_fields + whitespace_only) / len(frame)
                if len(frame) else float("nan")
            ),
            "possible_token_rows": int(stripped.isin(possible_missing_tokens).sum()),
        })
    print(name)
    display(pd.DataFrame(missing_rows).round({"missing_or_blank_pct": 2}))

crsp


,column,empty_fields,whitespace_only,missing_or_blank_pct,possible_token_rows
0,indno,0,0,0.0,0
1,mbrstartdt,0,0,0.0,0
2,mbrenddt,0,0,0.0,0
3,permno,0,0,0.0,0
4,hdrcusip,0,0,0.0,0
5,permco,0,0,0.0,0
6,cusip,31,0,0.0,0
7,ticker,31,0,0.0,0
8,dlycaldt,0,0,0.0,0


ccm


,column,empty_fields,whitespace_only,missing_or_blank_pct,possible_token_rows
0,gvkey,0,0,0.0,0
1,linkprim,0,0,0.0,0
2,liid,0,0,0.0,0
3,linktype,0,0,0.0,0
4,lpermno,0,0,0.0,0
5,lpermco,0,0,0.0,0
6,linkdt,0,0,0.0,0
7,linkenddt,0,0,0.0,0


compustat


,column,empty_fields,whitespace_only,missing_or_blank_pct,possible_token_rows
0,costat,0,0,0.00,0
1,curcd,0,0,0.00,0
2,datafmt,0,0,0.00,0
3,indfmt,0,0,0.00,0
4,consol,0,0,0.00,0
5,tic,24,0,0.02,6
6,datadate,0,0,0.00,0
7,gvkey,0,0,0.00,0
8,conm,0,0,0.00,0
9,cusip,24,0,0.02,0


revelio


,column,empty_fields,whitespace_only,missing_or_blank_pct,possible_token_rows
0,ticker,0,0,0.00,0
1,rcid,0,0,0.00,0
2,company,0,0,0.00,0
3,factset_entity_id,0,0,0.00,0
4,isin,0,0,0.00,0
5,cusip,2,0,0.18,0
6,url,10,0,0.89,0
7,naics_code,0,0,0.00,0
8,cik,426,0,37.90,0
9,lei,226,0,20.11,0


### 1.5 Basic duplicate diagnostics

An exact duplicate means all **loaded** column values repeat. For CRSP, this is a
nine-column projection, not a check of equality across all 42 source columns.

A repeated key is a different concept. The table reports both extra rows beyond the
first occurrence and all rows involved in repeated groups. Rows with any missing or
blank key component are counted separately and excluded from the key-duplicate check.
No rows are dropped from the input tables.

Compustat `GVKEY + FYEAR` and `GVKEY + DATADATE` need not be unique because formats and
other dimensions can differ. Even the expanded key below is a diagnostic, not an approved
record-selection rule. A repeated Revelio GVKEY or ticker does not by itself prove an
incorrect mapping.

In [8]:
exact_duplicate_rows = []
for name, frame in datasets.items():
    exact_duplicate_rows.append({
        "dataset": name,
        "columns_checked": len(frame.columns),
        "extra_identical_rows": int(frame.duplicated().sum()),
        "all_rows_in_identical_groups": int(frame.duplicated(keep=False).sum()),
    })

display(pd.DataFrame(exact_duplicate_rows))

,dataset,columns_checked,extra_identical_rows,all_rows_in_identical_groups
0,crsp,9,0,0
1,ccm,8,0,0
2,compustat,15,0,0
3,revelio,16,0,0


In [9]:
key_checks = [
    ("crsp", ["permno", "dlycaldt"]),
    ("ccm", ["lpermno", "gvkey", "liid", "linkdt", "linkenddt", "linkprim", "linktype"]),
    ("compustat", ["gvkey", "fyear"]),
    ("compustat", ["gvkey", "datadate"]),
    ("compustat", ["gvkey", "datadate", "indfmt", "datafmt", "consol", "curcd"]),
    ("revelio", ["rcid"]),
    ("revelio", ["gvkey"]),
    ("revelio", ["ticker"]),
]

key_rows = []
for name, columns in key_checks:
    keys = datasets[name][columns].copy()
    for column in columns:
        keys[column] = keys[column].str.strip()
    incomplete = keys.isna().any(axis=1) | keys.eq("").any(axis=1)
    complete_keys = keys.loc[~incomplete]
    key_rows.append({
        "dataset": name,
        "key": " + ".join(columns),
        "rows_with_incomplete_key": int(incomplete.sum()),
        "extra_rows_with_repeated_key": int(complete_keys.duplicated().sum()),
        "all_rows_with_repeated_key": int(complete_keys.duplicated(keep=False).sum()),
    })

display(pd.DataFrame(key_rows))

,dataset,key,rows_with_incomplete_key,extra_rows_with_repeated_key,all_rows_with_repeated_key
0,crsp,permno + dlycaldt,0,0,0
1,ccm,lpermno + gvkey + liid + linkdt + linkenddt + ...,0,0,0
2,compustat,gvkey + fyear,0,8179,16358
3,compustat,gvkey + datadate,0,8176,16352
4,compustat,gvkey + datadate + indfmt + datafmt + consol +...,0,0,0
5,revelio,rcid,0,0,0
6,revelio,gvkey,214,0,0
7,revelio,ticker,0,496,781


### 1.6 Date coverage and open-ended links

Parse ISO dates into temporary Series; retain the original date columns in the input
tables. Report missing/blank values, open-ended markers, and other unparseable values
separately. Minimum and maximum dates refer only to successfully parsed dates.

For CCM `LINKENDDT`, `E` means **open-ended**, not unknown historical coverage. No
replacement end date is applied in this Part. A proposed later overlap check would use
`study_end` as the effective analysis end for an `E` link while retaining its original
marker. That proposal requires review before implementation.

Dates outside the study window are not automatically errors: membership and link
histories can legitimately start earlier. Compustat `DATADATE` is a financial reporting
date, not a substitute for `FYEAR`. Daily coverage follows observed trading dates.

In [10]:
date_columns = {
    "crsp": ["dlycaldt", "mbrstartdt", "mbrenddt"],
    "ccm": ["linkdt", "linkenddt"],
    "compustat": ["datadate"],
}

date_rows = []
for name, columns in date_columns.items():
    for column in columns:
        values = datasets[name][column].str.strip()
        missing = values.isna() | values.eq("").fillna(False)
        open_ended = pd.Series(False, index=values.index)
        if name == "ccm" and column == "linkenddt":
            open_ended = values.eq("E").fillna(False)
        parsed_dates = pd.to_datetime(
            values.mask(open_ended), format="%Y-%m-%d", errors="coerce",
        )
        invalid = parsed_dates.isna() & ~missing & ~open_ended
        date_rows.append({
            "dataset": name,
            "column": column,
            "earliest_parsed_date": parsed_dates.min(),
            "latest_parsed_date": parsed_dates.max(),
            "missing_or_blank": int(missing.sum()),
            "open_ended_E": int(open_ended.sum()),
            "other_unparseable": int(invalid.sum()),
            "rows_before_study": int(parsed_dates.lt(study_start).sum()),
            "rows_after_study": int(parsed_dates.gt(study_end).sum()),
        })

display(pd.DataFrame(date_rows))

,dataset,column,earliest_parsed_date,latest_parsed_date,missing_or_blank,open_ended_E,other_unparseable,rows_before_study,rows_after_study
0,crsp,dlycaldt,2018-01-02,2025-12-31,0,0,0,0,0
1,crsp,mbrstartdt,1925-12-31,2025-12-22,0,0,0,872000,0
2,crsp,mbrenddt,2018-03-06,2025-12-31,0,0,0,0,0
3,ccm,linkdt,1949-07-01,2025-11-03,0,0,0,734,0
4,ccm,linkenddt,1962-01-01,2025-12-31,0,592,0,122,0
5,compustat,datadate,2018-01-31,2025-12-31,0,0,0,0,0


### 1.7 Source-specific aggregate checks

Review CCM link categories without filtering them. Review Compustat formats and fiscal
years without selecting an annual record. Count distinct RCID values per nonblank
Revelio ticker without matching any firms.

These diagnostics describe the downloaded extracts. In particular, Revelio coverage
cannot be inferred from ticker counts, and Compustat EMP missingness above describes
the full extract rather than the eventual historical research sample.

In [11]:
print("CRSP index identifiers")
display(crsp["indno"].value_counts(dropna=False).rename_axis("indno").to_frame("rows"))

for column in ["linktype", "linkprim"]:
    print(f"CCM: {column}")
    display(ccm[column].value_counts(dropna=False).rename_axis(column).to_frame("rows"))

for column in ["indfmt", "fyear"]:
    print(f"Compustat: {column}")
    display(
        compustat[column].value_counts(dropna=False)
        .sort_index().rename_axis(column).to_frame("rows")
    )

CRSP index identifiers


,rows
indno,
1000500,1013723


CCM: linktype


,rows
linktype,
LC,543
LU,242
LS,3


CCM: linkprim


,rows
linkprim,
P,667
C,108
J,11
N,2


Compustat: indfmt


,rows
indfmt,
FS,8176
INDL,99003


Compustat: fyear


,rows
fyear,
2017,970
2018,12440
2019,12647
2020,12966
2021,13315
2022,13612
2023,13698
2024,13913
2025,13618


In [12]:
revelio_ticker_keys = revelio[["ticker", "rcid"]].copy()
for column in revelio_ticker_keys.columns:
    revelio_ticker_keys[column] = revelio_ticker_keys[column].str.strip()

valid_ticker = (
    revelio_ticker_keys["ticker"].notna()
    & revelio_ticker_keys["ticker"].ne("")
)
valid_rcid = (
    revelio_ticker_keys["rcid"].notna()
    & revelio_ticker_keys["rcid"].ne("")
)
rcids_per_ticker = (
    revelio_ticker_keys.loc[valid_ticker & valid_rcid]
    .groupby("ticker")["rcid"].nunique()
)

display(pd.DataFrame([{
    "rows_excluded_for_missing_ticker_or_rcid": int((~(valid_ticker & valid_rcid)).sum()),
    "tickers_with_nonmissing_rcid": len(rcids_per_ticker),
    "tickers_with_multiple_rcids": int(rcids_per_ticker.gt(1).sum()),
}]))

,rows_excluded_for_missing_ticker_or_rcid,tickers_with_nonmissing_rcid,tickers_with_multiple_rcids
0,0,628,285


### Findings

- The CRSP extract contains 1,013,723 daily observations for 664 securities (PERMNO)
  and 657 companies (PERMCO), all under index identifier 1000500. Daily dates run from
  January 2, 2018 to December 31, 2025, with no repeated PERMNO-date keys. These are
  daily-data counts; membership-interval counts remain to be checked in Part 2.
- CCM contains 788 link records for 664 distinct LPERMNO. Of the 788 end-date values,
  592 are open-ended `E` markers. Equal security counts across CRSP and CCM do not
  establish identifier coverage or historically valid matching.
- Compustat contains 107,179 observations for 18,502 GVKEY, including 8,176 FS-format
  records. There are 8,176 extra rows beyond the first per GVKEY-date key, while the
  expanded key including format, consolidation, and currency has no repeats. EMP is
  blank in 46,878 of 107,179 rows (43.74%); this is extract-wide missingness, not
  research-sample missingness. No annual record-selection rule has been applied.
- Compustat reporting dates span January 31, 2018 to December 31, 2025, but 970 rows
  have fiscal year 2017. Reporting-date coverage and fiscal-year coverage must be
  distinguished when defining the subsequent annual panel.
- Revelio contains 1,124 distinct RCID, of which 910 have a nonblank GVKEY. Of 628
  distinct tickers, 285 correspond to multiple RCID. These results support further
  identifier validation; they do not measure historical-sample Revelio coverage.

## Part 2 - Historical S&P 500 Membership

One row represents a recorded membership interval for a security, not a company-year.
A PERMCO can have multiple PERMNO, so security and company counts need not agree.

Run **Part 0** setup before this Part. All diagnostics are computed from the file.
No records are removed, corrected, or exported. Detailed tables are for local review.

### 2.1 Membership structure

Retain the five source columns as strings with lowercase names. 

Add a row identifier for tracing exceptions; it numbers data rows from one, excluding the CSV header.
Exact duplicate counts use the five source columns, excluding this new identifier.

In [13]:
membership_path = project_root / "data/clean/sp500_membership_intervals.csv"
membership = pd.read_csv(
    membership_path, dtype="string", keep_default_na=False, na_values=[""],
)
membership.columns = membership.columns.str.lower()

membership_columns = ["indno", "mbrstartdt", "mbrenddt", "permno", "permco"]
missing_columns = set(membership_columns) - set(membership.columns)
if missing_columns:
    raise ValueError(f"Missing membership columns: {sorted(missing_columns)}")

membership["membership_row"] = range(1, len(membership) + 1)

# Trim identifiers only in a diagnostic copy; preserve the source columns.
membership_identifiers = membership[["indno", "permno", "permco"]].copy()
for column in membership_identifiers.columns:
    membership_identifiers[column] = (
        membership_identifiers[column].str.strip().replace("", pd.NA)
    )

membership["missing_identifier"] = membership_identifiers.isna().any(axis=1)
membership["unexpected_index"] = (
    membership_identifiers["indno"].ne("1000500").fillna(False)
)
membership["exact_duplicate"] = membership.duplicated(
    subset=membership_columns, keep=False,
)

In [14]:
membership_summary = pd.Series({
    "membership_interval_records": len(membership),
    "distinct_source_records": int(
        (~membership.duplicated(subset=membership_columns)).sum()
    ),
    "unique_index_identifiers": membership_identifiers["indno"].nunique(),
    "distinct_permno": membership_identifiers["permno"].nunique(),
    "distinct_permco": membership_identifiers["permco"].nunique(),
    "rows_with_missing_identifiers": int(membership["missing_identifier"].sum()),
    "rows_in_exact_duplicate_groups": int(membership["exact_duplicate"].sum()),
}, name="count")
display(membership_summary.to_frame())
display(
    membership_identifiers["indno"].value_counts(dropna=False)
    .rename_axis("indno").to_frame("interval_records")
)

,count
membership_interval_records,666
distinct_source_records,666
unique_index_identifiers,1
distinct_permno,664
distinct_permco,657
rows_with_missing_identifiers,0
rows_in_exact_duplicate_groups,0


,interval_records
indno,
1000500,666


### 2.2 Repeated membership

Identify every nonmissing PERMNO with more than one record. Repeated records may
represent separate membership episodes or duplicate/overlapping intervals; inspect
the dates before interpreting them as re-entry. Retain every original row.

Record-level previews are limited to 30 rows to keep saved outputs compact. The full
DataFrames remain available locally; a printed total identifies any truncated preview.

In [15]:
intervals_per_permno = membership_identifiers.groupby("permno").size()
repeated_permnos = intervals_per_permno.loc[intervals_per_permno.gt(1)].index
membership["repeated_permno"] = membership_identifiers["permno"].isin(repeated_permnos)

review_columns = ["membership_row", "indno", "permno", "permco", "mbrstartdt", "mbrenddt"]
repeated_membership = membership.loc[
    membership["repeated_permno"], review_columns,
].sort_values(["permno", "mbrstartdt", "mbrenddt"])

print(f"PERMNO with multiple interval records: {len(repeated_permnos)}")
print(f"Repeated-membership records: {len(repeated_membership)}; preview limit: 30")
display(repeated_membership.head(30))

PERMNO with multiple interval records: 2
Repeated-membership records: 4; preview limit: 30


,membership_row,indno,permno,permco,mbrstartdt,mbrenddt
41,42,1000500,13688,21356,1925-12-31,2019-01-17
604,605,1000500,13688,21356,2022-10-03,2025-12-31
130,131,1000500,24328,20662,2008-12-19,2018-11-12
605,606,1000500,24328,20662,2022-10-03,2025-12-31


### 2.3 Date validation

Parse dates into new columns, leaving `mbrstartdt` and `mbrenddt` unchanged. Separate
missing/blank dates from nonblank values that cannot be parsed. Reversed intervals
have a parsed start later than the parsed end and are not eligible for overlap tests.

Flag starts on or before the previously noted date `1925-12-31` for interpretation.
This is a review flag, not a correction rule or a claim that these dates are invalid.
Also inspect the ten earliest distinct parsed start dates for other unusual values.

In [16]:
membership_date_rows = []
for source_column, parsed_column in [
    ("mbrstartdt", "membership_start"),
    ("mbrenddt", "membership_end"),
]:
    source_dates = membership[source_column].str.strip()
    missing_dates = source_dates.isna() | source_dates.eq("").fillna(False)
    membership[parsed_column] = pd.to_datetime(
        source_dates, format="%Y-%m-%d", errors="coerce",
    )
    invalid_dates = membership[parsed_column].isna() & ~missing_dates
    membership[f"{parsed_column}_missing"] = missing_dates
    membership[f"{parsed_column}_invalid"] = invalid_dates
    membership_date_rows.append({
        "column": source_column,
        "missing_or_blank": int(missing_dates.sum()),
        "nonblank_unparseable": int(invalid_dates.sum()),
        "earliest_parsed": membership[parsed_column].min(),
        "latest_parsed": membership[parsed_column].max(),
    })

membership["reversed_interval"] = (
    membership["membership_start"] > membership["membership_end"]
)
membership["valid_dates"] = (
    membership[["membership_start", "membership_end"]].notna().all(axis=1)
    & ~membership["reversed_interval"]
)
membership["early_start_for_review"] = (
    membership["membership_start"] <= pd.Timestamp("1925-12-31")
)

display(pd.DataFrame(membership_date_rows))
print(f"Reversed intervals: {int(membership['reversed_interval'].sum())}")
display(
    membership["membership_start"].value_counts().sort_index().head(10)
    .rename_axis("earliest_start_dates").to_frame("interval_records")
)

,column,missing_or_blank,nonblank_unparseable,earliest_parsed,latest_parsed
0,mbrstartdt,0,0,1925-12-31,2025-12-22
1,mbrenddt,0,0,2018-03-06,2025-12-31


Reversed intervals: 0


,interval_records
earliest_start_dates,
1925-12-31,10
1944-06-07,2
1948-07-07,1
1949-07-06,1
1951-09-06,1
1957-03-01,34
1957-07-10,1
1959-03-25,1
1959-04-01,1


### 2.4 Study-window boundaries

Use inclusive calendar-date boundaries: an interval overlaps the study period when
its start is on or before `study_end` and its end is on or after `study_start`.
Intervals with missing, invalid, or reversed dates remain unassessable.

For valid overlapping intervals, `effective_start` is the later of the historical
start and `study_start`; `effective_end` is the earlier of the historical end and
`study_end`. Other records retain missing effective dates and remain in the table.
These columns describe observation within the study window, not revised membership.
An original end date equal to `2025-12-31` does not establish an actual index deletion.
Counts below are interval-record counts; the boundary flags may overlap.

In [17]:
membership["overlaps_study"] = (
    membership["valid_dates"]
    & membership["membership_start"].le(study_end)
    & membership["membership_end"].ge(study_start)
)
membership["outside_study"] = membership["valid_dates"] & ~membership["overlaps_study"]
membership["ends_at_study_boundary"] = membership["membership_end"].eq(study_end)

membership["effective_start"] = (
    membership["membership_start"].clip(lower=study_start)
    .where(membership["overlaps_study"])
)
membership["effective_end"] = (
    membership["membership_end"].clip(upper=study_end)
    .where(membership["overlaps_study"])
)

window_summary = pd.Series({
    "total_interval_records": len(membership),
    "overlapping_study_window": int(membership["overlaps_study"].sum()),
    "entirely_outside_study_window": int(membership["outside_study"].sum()),
    "unassessable_dates": int((~membership["valid_dates"]).sum()),
    "overlapping_intervals_starting_before_window": int((
        membership["overlaps_study"] & membership["membership_start"].lt(study_start)
    ).sum()),
    "overlapping_intervals_ending_after_window": int((
        membership["overlaps_study"] & membership["membership_end"].gt(study_end)
    ).sum()),
    "original_end_equals_study_end": int(membership["ends_at_study_boundary"].sum()),
}, name="interval_records")
display(window_summary.to_frame())

,interval_records
total_interval_records,666
overlapping_study_window,666
entirely_outside_study_window,0
unassessable_dates,0
overlapping_intervals_starting_before_window,504
overlapping_intervals_ending_after_window,0
original_end_equals_study_end,503


### 2.5 Overlapping intervals for the same security

Compare all distinct pairs of valid intervals within each repeated PERMNO. This small
within-table comparison also detects nested intervals; comparing only adjacent end
dates could miss them. A shared endpoint counts as overlap under inclusive boundaries.
Pairs use original historical dates, not effective dates. Exact duplicate intervals
can also appear here. Missing PERMNO and invalid/reversed intervals are excluded only
from this diagnostic and remain in `membership` for review.

In [18]:
comparable_membership = membership.loc[
    membership["repeated_permno"] & membership["valid_dates"],
    ["membership_row", "membership_start", "membership_end"],
].copy()
comparable_membership["permno"] = membership_identifiers.loc[
    comparable_membership.index, "permno",
]

interval_pairs = comparable_membership.merge(
    comparable_membership, on="permno", suffixes=("_a", "_b"),
)
# Keep each pair once, excluding a record paired with itself.
interval_pairs = interval_pairs.loc[
    interval_pairs["membership_row_a"] < interval_pairs["membership_row_b"]
].copy()
overlapping_intervals = interval_pairs.loc[
    interval_pairs["membership_start_a"].le(interval_pairs["membership_end_b"])
    & interval_pairs["membership_start_b"].le(interval_pairs["membership_end_a"])
].copy()

membership["overlapping_interval"] = (
    membership["membership_row"].isin(overlapping_intervals["membership_row_a"])
    | membership["membership_row"].isin(overlapping_intervals["membership_row_b"])
)
print(f"Overlapping interval pairs: {len(overlapping_intervals)}; preview limit: 30")
print(f"PERMNO with overlapping intervals: {overlapping_intervals['permno'].nunique()}")
display(overlapping_intervals.head(30))

Overlapping interval pairs: 0; preview limit: 30
PERMNO with overlapping intervals: 0


,membership_row_a,membership_start_a,membership_end_a,permno,membership_row_b,membership_start_b,membership_end_b


### 2.6 Exceptions for local review

Combine review flags into one table without removing any membership records. An
exception flag requests interpretation; it does not by itself imply an erroneous row.
Sample-end dates are reported separately in the boundary summary rather than labeling
all such records as errors. The combined table includes original and effective dates.
Review each flag's total and the preview; all flagged rows remain in
`membership_exceptions` if more than 30 records require inspection.

In [19]:
review_flags = [
    "missing_identifier", "unexpected_index", "exact_duplicate", "repeated_permno",
    "membership_start_missing", "membership_start_invalid",
    "membership_end_missing", "membership_end_invalid",
    "reversed_interval", "outside_study", "early_start_for_review",
    "overlapping_interval",
]

display(membership[review_flags].sum().rename("flagged_records").to_frame())

membership["review_reason"] = ""
for flag in review_flags:
    flagged = membership[flag]
    membership.loc[flagged, "review_reason"] = (
        membership.loc[flagged, "review_reason"] + flag + "; "
    )
membership["review_reason"] = membership["review_reason"].str.rstrip("; ")

exception_columns = review_columns + ["effective_start", "effective_end", "review_reason"]
membership_exceptions = membership.loc[
    membership[review_flags].any(axis=1), exception_columns,
].sort_values(["permno", "mbrstartdt", "membership_row"])

print(f"Records requiring review: {len(membership_exceptions)}; preview limit: 30")
with pd.option_context("display.max_columns", None, "display.max_colwidth", 120):
    display(membership_exceptions.head(30))

,flagged_records
missing_identifier,0
unexpected_index,0
exact_duplicate,0
repeated_permno,4
membership_start_missing,0
membership_start_invalid,0
membership_end_missing,0
membership_end_invalid,0
reversed_interval,0
outside_study,0


Records requiring review: 13; preview limit: 30


,membership_row,indno,permno,permco,mbrstartdt,mbrenddt,effective_start,effective_end,review_reason
3,4,1000500,10145,22168,1925-12-31,2025-12-31,2018-01-01,2025-12-31,early_start_for_review
9,10,1000500,11404,20499,1925-12-31,2025-12-31,2018-01-01,2025-12-31,early_start_for_review
13,14,1000500,11674,20574,1925-12-31,2025-12-31,2018-01-01,2025-12-31,early_start_for_review
15,16,1000500,11850,20678,1925-12-31,2025-12-31,2018-01-01,2025-12-31,early_start_for_review
41,42,1000500,13688,21356,1925-12-31,2019-01-17,2018-01-01,2019-01-17,repeated_permno; early_start_for_review
604,605,1000500,13688,21356,2022-10-03,2025-12-31,2022-10-03,2025-12-31,repeated_permno
57,58,1000500,14541,20440,1925-12-31,2025-12-31,2018-01-01,2025-12-31,early_start_for_review
65,66,1000500,15069,21795,1925-12-31,2024-11-25,2018-01-01,2024-11-25,early_start_for_review
72,73,1000500,15720,21543,1925-12-31,2025-12-31,2018-01-01,2025-12-31,early_start_for_review
76,77,1000500,16432,20834,1925-12-31,2019-02-26,2018-01-01,2019-02-26,early_start_for_review


### Findings

- The table contains 666 distinct membership intervals for 664 PERMNO and 657 PERMCO, all under index identifier 1000500. No missing identifiers or exact duplicate records were found.
- Two securities each have two non-overlapping intervals. These records are consistent with separate membership episodes and were retained without collapsing them.
- All 666 intervals overlap the 2018-2025 study window. Dates are complete and parseable, with no reversed intervals or overlapping periods for the same security. Effective starts were limited to January 1, 2018 for 504 intervals; original dates were preserved.
- Ten intervals begin on December 31, 1925 and require historical interpretation. Another boundary issue concerns the 503 intervals ending on December 31, 2025: this sample-end date should not be treated as a confirmed index deletion.
- The review table contains 13 distinct records: 10 with early starts and four belonging to repeated-membership securities, with one record in both groups. All records remain available for subsequent CCM linking; no dates or membership episodes were corrected or removed.